<a href="https://colab.research.google.com/github/santoryuZ/open_clip_benchmark/blob/main/miniLM_benchmark.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install sentence-transformers pdfplumber python-docx pandas scikit-learn seaborn matplotlib tqdm -q

###Importing dependencies

In [ ]:
import os, time, csv
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix
from tqdm import tqdm
from pathlib import Path

print("Ready")

###Setting up Dataset

In [ ]:
!unzip -q "/content/document_dataset.zip" -d "/content/"

In [ ]:
import os

zip_file_path = "/content/document_dataset.zip"

if os.path.exists(zip_file_path):
    file_size = os.path.getsize(zip_file_path)
    print(f"The size of {zip_file_path} is {file_size} bytes.")
    if file_size == 0:
        print("WARNING: The file is 0 bytes, indicating it is empty or corrupted.")
else:
    print(f"ERROR: The file {zip_file_path} does not exist.")

In [ ]:
import os
import shutil

# Clean up any previous attempts or corrupted files
if os.path.exists('/content/document_dataset.zip'):
  os.remove('/content/document_dataset.zip')
if os.path.exists('/content/document_dataset'):
  shutil.rmtree('/content/document_dataset')

# Download a fresh copy of the dataset
# This URL points to the dataset used in the original example from huggingface/docquery
print("Attempting to download document_dataset.zip...")
# The previous URL was returning a 404 Not Found error. Please update this URL with a working one.
DATASET_DOWNLOAD_URL = "https://github.com/huggingface/docquery/raw/main/examples/document_dataset.zip"
!wget {DATASET_DOWNLOAD_URL} -O /content/document_dataset.zip

# Check if download was successful and file is not empty
if not os.path.exists('/content/document_dataset.zip') or os.path.getsize('/content/document_dataset.zip') == 0:
    print("ERROR: document_dataset.zip was not downloaded successfully or is empty. Please check the download URL.")
else:
    print(f"Downloaded /content/document_dataset.zip (size: {os.path.getsize('/content/document_dataset.zip')} bytes)")
    # Unzip the newly downloaded dataset
    print("Attempting to unzip document_dataset.zip...")
    # Keep -q for unzip for cleaner output unless it needs debugging too
    !unzip -q /content/document_dataset.zip
    # Check if unzip was successful (by checking for the dataset directory)
    if os.path.exists('/content/document_dataset'):
        print("Dataset downloaded and unzipped successfully.")
    else:
        print("ERROR: Failed to unzip document_dataset.zip. The file might be corrupted or unzip failed.")

In [ ]:
DATASET_PATH = "/content/document_dataset"

CATEGORIES = sorted([
    f for f in os.listdir(DATASET_PATH)
    if os.path.isdir(os.path.join(DATASET_PATH, f))
])

all_file_paths = []
all_true_labels = []

for cat in CATEGORIES:
  folder = os.path.join(DATASET_PATH, cat)
  files = [
      f for f in os.listdir(folder)
      if f.lower().endswith((".pdf", ".docx", ".txt"))
  ]
  for file in files:
    all_file_paths.append(os.path.join(folder, file))
    all_true_labels.append(cat)
  print(f" {cat}: {len(files)} files")

print(f"\n Total: {len(all_file_paths)} documents across {len(CATEGORIES)} categories")

##Text Extraction

In [ ]:
import pdfplumber
import docx

def extract_text(file_path: str, max_chars: int = 2000) -> str:
  """
  Extract raw text from a pdf, docx or txt file.
  returns empty string if extraction fails or file is corrupter or has not text
  max_char limits the number of text passed to the model - the first 2000 chars
  are enough for classification, keeping inference fast
  """

  path = Path(file_path)

  try:
    if path.suffix.lower() == ".pdf":
      chunks = []
      with pdfplumber.open(path) as pdf:
        for page in pdf.pages[:5]:
          text = page.extract_text()
          if text:
            chunks.append(text)
        return "\n".join(chunks)[:max_chars]

    elif path.suffix.lower() == ".docx":
      doc = docx.Document(path)
      text = "\n".join([para.text for para in doc.paragraphs])
      return text[:max_chars]

    elif path.suffix.lower() == ".txt":
      with open(path, 'r', encoding='utf-8', errors='ignore') as f:
        return f.read(max_chars)

  except Exception as e:
      print(f"Could not extract: {path.name} ({e})")
  return ""

print("Extracting text and filtering dataset...")
filtered_texts = []
filtered_paths = []
filtered_labels = []
skipped = []

for path, label in tqdm(zip(all_file_paths, all_true_labels), total=len(all_file_paths)):
  text = extract_text(path)
  if text.strip():
    filtered_texts.append(text)
    filtered_paths.append(path)
    filtered_labels.append(label)
  else:
    skipped.append(Path(path).name)

# Overwrite global lists to only include successful extractions
all_texts = filtered_texts
all_file_paths = filtered_paths
all_true_labels = filtered_labels

print(f"\n Done. {len(all_texts)} documents kept, {len(skipped)} files skipped (no extractable text).")

if skipped:
  print("Skipped files list:")
  for f in skipped:
    print(f" - {f}")

##Saving Prompts


In [ ]:
PROMPTS = {
    "resume": [
        "a resume or curriculum vitae listing work experience, education history, and professional skills",
        "a CV or resume document with sections for employment history, academic qualifications, and contact details",
        "a job application document showing a candidate's career history, skills, and educational background",
        "a professional resume with structured sections including work experience, education, and skill set",
    ],
    "invoice": [
        "an invoice or bill listing purchased items or services with individual prices and a total amount due",
        "a payment invoice with line items, quantities, unit costs, subtotal, tax, and grand total",
        "a financial invoice document from a business showing itemised charges and payment terms",
        "a billing document or commercial invoice with product descriptions, prices, and a payment total",
    ],
    "legal": [
        "a legal contract or formal agreement between two or more parties with numbered clauses and signature blocks",
        "a legal document such as a contract, agreement, or terms of service with binding obligations and conditions",
        "a formal legal agreement with definitions, obligations, representations, and signature sections",
        "a contract document containing legal language, party names, numbered articles, and execution blocks",
    ],
    "general": [
        "a general office document such as a business letter, memo, report, or formal correspondence",
        "an official letter or formal office document with a header, body paragraphs, and a closing signature",
        "a standard business document such as a report, policy document, or formal letter without itemised pricing",
        "a written document or correspondence that does not follow an invoice, resume, or legal contract format",
    ],
}

# Sanity check
missing = [c for c in CATEGORIES if c not in PROMPTS]
if missing:
    print(f"WARNING: no prompts for: {missing} — add them before continuing")
else:
    print("All categories have prompts.")

##Models to test

In [ ]:
MODELS_TO_TEST = [
    "all-MiniLM-L6-v2",
    "all-MiniLM-L12-v2",
    "all-mpnet-base-v2",
    "all-distilroberta-v1",
]

print(f"Testing {len(MODELS_TO_TEST)} models")

#Benchmark Function

In [ ]:
from sentence_transformers import SentenceTransformer

def benchmark_text_model(model_name, texts, true_labels, categories, prompts):
  print(f"\n{'='*50}")
  print(f"Testing model: {model_name}")
  print(f"{'='*50}")

  #Loading model
  t0 = time.time()
  model = SentenceTransformer(model_name)
  print(f"Loaded in {time.time()-t0:.1f}s")

  #We need BGE prefix because BGE models were trained with specific query for retrieval
  bge_prefix = "Represent this sentence for searching relevant passages: "
  is_bge = model_name.startswith("bge-")


  #Encoding category prompts with ensmeble averaging
  #Each category gets 4 prompt variants; we average their embeddings
  prompt_embeddings = []
  for cat in categories:
    variants = prompts[cat]
    if is_bge:
      variants = [bge_prefix + v for v in variants]
    vecs = model.encode(variants, normalize_embeddings=True, show_progress_bar=False)
    avg = vecs.mean(axis=0)
    avg /= np.linalg.norm(avg)
    prompt_embeddings.append(avg)
  prompt_embeddings = np.array(prompt_embeddings)

  # Running inference on every document
  predicted_labels = []
  confidence_scores = []
  inference_times = []
  skipped_indices = []


  for i, (text, path) in enumerate(tqdm(zip(texts, all_file_paths), total=len(texts), desc=model_name)):
    if not text.strip():
      # No text extracted
      predicted_labels.append("NO_TEXT")
      confidence_scores.append(0.0)
      inference_times.append(0.0)
      skipped_indices.append(i)
      continue

    prefix = bge_prefix if is_bge else ""
    t0 = time.time()
    doc_embedding = model.encode(prefix + text, normalize_embeddings=True, show_progress_bar=False)
    elapsed = time.time() - t0


    #Raw cosine similarity - dot product of two L2-normalized vectors.
    similarities = prompt_embeddings @ doc_embedding
    best_idx = int(similarities.argmax())
    predicted_labels.append(categories[best_idx])
    confidence_scores.append(float(similarities[best_idx]))
    inference_times.append(elapsed*1000)

  # Seperate confidence for correct vs wrong
  correct_conf = [
      c for p, t, c in zip(predicted_labels, true_labels, confidence_scores)
      if p == t
  ]
  wrong_conf = [
      c for p, t, c in zip(predicted_labels, true_labels, confidence_scores)
      if p != t and p != "NO_TEXT"
  ]

  valid_pred = [p for p in predicted_labels if p != "NO _TEXT"]
  valid_true = [t for p, t in zip(predicted_labels, true_labels) if p != "NO_TEXT"]
  valid_times = [t for t in inference_times if t>0]

  accuracy = sum(p == t for p, t in zip(valid_pred, valid_true)) / len(valid_true) * 100


  print(f"Overall accuracy:         {accuracy:.2f}%  ({sum(p==t for p,t in zip(valid_pred,valid_true))}/{len(valid_true)})")
  print(f"Avg inference time:       {np.mean(valid_times):.1f} ms/doc")
  print(f"Avg confidence (correct): {np.mean(correct_conf):.3f}")
  print(f"Avg confidence (wrong):   {np.mean(wrong_conf):.3f}")
  print(f"Confidence gap:           {np.mean(correct_conf) - np.mean(wrong_conf):.3f}  ← this is the key number")
  if skipped_indices:
      print(f"Skipped (no text):        {len(skipped_indices)} files")

  print(f"\nPer-category accuracy:")
  print(classification_report(valid_true, valid_pred, target_names=categories, digits=3))

  del model

  return {
      "model" : model_name,
      "predicted_labels": predicted_labels,
      "confidence_scores": confidence_scores,
      "inference_times": inference_times,
      "avg_ms": round(np.mean(valid_times), 1),
      "accuracy": round(accuracy, 2),
      "avg_conf_correct": round(np.mean(correct_conf), 3),
      "avg_conf_wrong": round(np.mean(wrong_conf), 3),
      "conf_gap" : round(np.mean(correct_conf) - np.mean(wrong_conf), 3)
  }


In [ ]:
#Saving result
all_results = []

for model_name in MODELS_TO_TEST:
  result = benchmark_text_model(
      model_name = model_name,
      texts = all_texts,
      true_labels = all_true_labels,
      categories = CATEGORIES,
      prompts = PROMPTS
  )

  all_results.append(result)
print("All models done.")


## Confusion Matrix

In [ ]:
# Summary table
summary = pd.DataFrame([{
    "Model":            r["model"],
    "Accuracy %":       r["accuracy"],
    "Avg ms/doc":       r["avg_ms"],
    "Conf correct":     r["avg_conf_correct"],
    "Conf wrong":       r["avg_conf_wrong"],
    "Conf gap":         r["conf_gap"],
} for r in all_results]).sort_values("Accuracy %", ascending=False)

print(summary.to_string(index=False))

# Confusion matrices
fig, axes = plt.subplots(1, len(all_results), figsize=(7 * len(all_results), 6))
if len(all_results) == 1:
    axes = [axes]

for ax, result in zip(axes, all_results):
    valid = [(p, t) for p, t in zip(result["predicted_labels"], all_true_labels)
             if p != "NO_TEXT"]
    preds = [v[0] for v in valid]
    trues = [v[1] for v in valid]

    cm = confusion_matrix(trues, preds, labels=CATEGORIES)
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=CATEGORIES, yticklabels=CATEGORIES, ax=ax)
    ax.set_title(result["model"], fontsize=10)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")
    ax.tick_params(axis="x", rotation=40)

plt.tight_layout()
plt.savefig("/content/text_model_confusion_matrices.png", dpi=150, bbox_inches="tight")
plt.show()